# AI Programming — Lecture 19
## Lab 4-3: Transformer Encoder for Sequence-based Inverse Kinematics

Transformer Encoder를 **연속값 sequence**에 적용합니다.

입력은 30개의 end-effector pose sequence이고,
마지막 시점의 robot joint configuration을 예측합니다.

```text
[p1, p2, ..., p30]
        ↓
Transformer Encoder
        ↓
last encoder representation
        ↓
Dense regression head
        ↓
[q1, q2, q3, q4, q5, q6]
```

### 중요한 점
이 dataset의 연속 행은 실제로 기록된 robot trajectory를 의미하지 않습니다.
여기서는 **continuous sequence를 Transformer Encoder에 넣는 연습**으로 사용합니다.

### 학습 목표
- Continuous feature를 embedding dimension으로 projection할 수 있습니다.
- Sinusoidal positional encoding을 continuous sequence에 적용할 수 있습니다.
- Transformer Encoder를 many-to-one regression에 사용할 수 있습니다.
- 마지막 encoder representation을 regression head에 연결할 수 있습니다.

### Colab 데이터 경로
```text
MyDrive/Colab Notebooks/data/datasetIRB2400.csv
```

In [ ]:
# 1. Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
keras.utils.set_random_seed(42)

In [ ]:
# 2. Load robot data
from google.colab import drive
drive.mount("/content/drive")

csv_path = "/content/drive/MyDrive/Colab Notebooks/data/datasetIRB2400.csv"
df = pd.read_csv(csv_path)

print("Data shape:", df.shape)
df.head()

In [ ]:
# 3. Input and target
pose_columns = ["x", "y", "z", "yaw", "pitch", "roll"]
joint_columns = ["q1_out", "q2_out", "q3_out", "q4_out", "q5_out", "q6_out"]

X = df[pose_columns].values.astype("float32")
Y = df[joint_columns].values.astype("float32")

print("X:", X.shape)
print("Y:", Y.shape)

### Chronological Split

데이터를 순서대로

```text
Train 70% / Validation 15% / Test 15%
```

로 분리합니다.

Scaler는 **training data에만 fit**합니다.

In [ ]:
# 4. Chronological split: 70% / 15% / 15%
n = len(X)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train_raw, Y_train_raw = X[:train_end], Y[:train_end]
X_val_raw, Y_val_raw = X[train_end:val_end], Y[train_end:val_end]
X_test_raw, Y_test_raw = X[val_end:], Y[val_end:]

print(len(X_train_raw), len(X_val_raw), len(X_test_raw))

In [ ]:
# 5. Standardization: fit on training data only
scaler_X = StandardScaler()
scaler_Y = StandardScaler()

X_train = scaler_X.fit_transform(X_train_raw).astype("float32")
Y_train = scaler_Y.fit_transform(Y_train_raw).astype("float32")

X_val = scaler_X.transform(X_val_raw).astype("float32")
Y_val = scaler_Y.transform(Y_val_raw).astype("float32")

X_test = scaler_X.transform(X_test_raw).astype("float32")
Y_test = scaler_Y.transform(Y_test_raw).astype("float32")

In [ ]:
# 6. Create pose sequences
TIME_STEPS = 30

def create_sequences(X, Y, time_steps):
    Xs, Ys = [], []

    for i in range(len(X) - time_steps + 1):
        Xs.append(X[i:i + time_steps])
        Ys.append(Y[i + time_steps - 1])

    return np.array(Xs, dtype="float32"), np.array(Ys, dtype="float32")

X_train, Y_train = create_sequences(X_train, Y_train, TIME_STEPS)
X_val, Y_val = create_sequences(X_val, Y_val, TIME_STEPS)
X_test, Y_test = create_sequences(X_test, Y_test, TIME_STEPS)

print("Train:", X_train.shape, Y_train.shape)
print("Val  :", X_val.shape, Y_val.shape)
print("Test :", X_test.shape, Y_test.shape)

## Sinusoidal Positional Encoding

Pose 자체에는 sequence 위치 정보가 없으므로
sin/cos 기반 positional encoding을 더합니다.

In [ ]:
# 7. Sinusoidal positional encoding
class SinusoidalPositionalEncoding(layers.Layer):
    def __init__(self, max_len, embed_dim):
        super().__init__()

        positions = np.arange(max_len)[:, np.newaxis]
        div_term = np.exp(
            np.arange(0, embed_dim, 2) * (-np.log(10000.0) / embed_dim)
        )

        pe = np.zeros((max_len, embed_dim), dtype="float32")
        pe[:, 0::2] = np.sin(positions * div_term)
        pe[:, 1::2] = np.cos(positions * div_term)

        self.pe = keras.ops.convert_to_tensor(pe[np.newaxis, ...])

    def call(self, inputs):
        return inputs + self.pe

## Transformer Encoder

각 encoder block은 다음 요소로 구성됩니다.

```text
Multi-Head Self-Attention
→ Add & Norm
→ Feed Forward Network
→ Add & Norm
```

두 개의 encoder block을 사용합니다.

In [ ]:
# 8. Transformer Encoder block
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout
        )

        self.dense1 = layers.Dense(ff_dim, activation="relu")
        self.dense2 = layers.Dense(embed_dim)

        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()
        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, inputs):
        attention_output = self.attention(inputs, inputs)
        x = self.norm1(inputs + self.dropout1(attention_output))

        ffn_output = self.dense2(self.dense1(x))
        return self.norm2(x + self.dropout2(ffn_output))

In [ ]:
# 9. Build model
EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 128

inputs = keras.Input(shape=(TIME_STEPS, 6))

projection_layer = layers.Dense(EMBED_DIM)
x = projection_layer(inputs)

position_layer = SinusoidalPositionalEncoding(TIME_STEPS, EMBED_DIM)
x = position_layer(x)

encoder1 = TransformerEncoder(EMBED_DIM, NUM_HEADS, FF_DIM)
x = encoder1(x)

encoder2 = TransformerEncoder(EMBED_DIM, NUM_HEADS, FF_DIM)
x = encoder2(x)

last_token_layer = layers.Lambda(lambda x: x[:, -1, :])
x = last_token_layer(x)

dense_layer = layers.Dense(64, activation="relu")
x = dense_layer(x)

dropout_layer = layers.Dropout(0.1)
x = dropout_layer(x)

output_layer = layers.Dense(6)
outputs = output_layer(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="mse",
    metrics=["mae"]
)

model.summary()

In [ ]:
# 10. Train
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    Y_train,
    validation_data=(X_val, Y_val),
    epochs=100,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
# 11. Learning curve
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.show()

In [ ]:
# 12. Test prediction in original joint-angle scale
Y_pred_scaled = model.predict(X_test, verbose=0)

Y_pred = scaler_Y.inverse_transform(Y_pred_scaled)
Y_true = scaler_Y.inverse_transform(Y_test)

mae = mean_absolute_error(Y_true, Y_pred)
print(f"Overall MAE: {mae:.4f}")

mae_each = np.mean(np.abs(Y_true - Y_pred), axis=0)
for name, value in zip(joint_columns, mae_each):
    print(f"{name}: {value:.4f}")

In [ ]:
# 13. Example: q1 prediction
joint_index = 0

plt.plot(Y_true[:, joint_index], label="True")
plt.plot(Y_pred[:, joint_index], label="Predicted")
plt.xlabel("Test sample")
plt.ylabel(joint_columns[joint_index])
plt.legend()
plt.show()

## 정리

### 전체 흐름
```text
Pose sequence (30 × 6)
→ Dense projection
→ Positional encoding
→ Transformer Encoder × 2
→ Last token representation
→ Dense(64)
→ Joint angles (6)
```

### 확인할 내용
1. Text와 달리 continuous input에서는 `Embedding` 대신 왜 `Dense projection`을 사용할까요?
2. 왜 마지막 encoder representation만 사용하나요?
3. Training loss는 계속 감소하지만 validation loss가 증가한다면 어떤 현상일까요?
4. 어떤 joint에서 MAE가 가장 큰지 확인해 보세요.